<a href="https://colab.research.google.com/github/tburleyinfo/vLLM-Hook/blob/sandbox/notebooks/demo_actsteer_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Activation Steering

vLLM-Hook is an extensible framework that aims to allow selective access to model internals during inference.
In this notebook, we demonstrate how vLLM-Hook enables **Activation Steering** for controlled generation.

**Paper**: [Improving Instruction-Following in Language Models through Activation Steering](https://arxiv.org/abs/2410.12877).<br />
**Authors**: Alessandro Stolfo, Vidhisha Balachandran, Safoora Yousefi, Eric Horvitz, Besmira Nushi <br />
**"TL;DR"**: Activation steering allows you to bias the model's behavior by nudging internal activations in specific directions. In this paper, authors focus on instruction following capability and compute the steering vectors as the difference in activations between inputs with and without instructions. 

### Installation

If running this from a new environment, please use the cell below to install `vllm_hook_plugins`. Update the path/command to match your environment.<br />
The following block is not necessary if running this notebook from an environment where the package has already been installed.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import importlib.util

REPO_URL = "https://github.com/tburleyinfo/vLLM-Hook.git"
REPO_BRANCH = "sandbox"
REPO_NAME = "vLLM-Hook"

IN_COLAB = "google.colab" in sys.modules
NOTEBOOK_DIR = Path.cwd()


def _repo_remote_matches(repo_root: Path, expected_remote: str) -> bool:
    """Return True when repo_root points at the expected git remote."""
    try:
        origin_url = subprocess.run(
            ["git", "-C", str(repo_root), "remote", "get-url", "origin"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip().removesuffix('.git')
    except Exception:
        return False
    return origin_url == expected_remote


def _find_existing_repo_root(start_dir: Path, expected_remote: str):
    """Walk upward from start_dir and return a matching repo root when one exists."""
    for candidate in [start_dir, *start_dir.parents]:
        if (candidate / ".git").exists() and _repo_remote_matches(candidate, expected_remote):
            return candidate
    return None


if IN_COLAB:
    expected_remote = REPO_URL.removesuffix('.git')
    existing_repo_root = _find_existing_repo_root(NOTEBOOK_DIR, expected_remote)
    if existing_repo_root is not None:
        REPO_ROOT = existing_repo_root
        print(f"Colab detected. Reusing existing repo at {REPO_ROOT}")
    else:
        REPO_ROOT = Path('/content') / REPO_NAME
        if not REPO_ROOT.exists():
            print(f"Colab detected. Cloning {REPO_URL} ({REPO_BRANCH}) into {REPO_ROOT} ...")
            subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
        elif not _repo_remote_matches(REPO_ROOT, expected_remote):
            print(f"Remote mismatch under {REPO_ROOT}; replacing clone with {expected_remote}")
            shutil.rmtree(REPO_ROOT)
            subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
        else:
            print(f"Colab detected. Reusing existing clone at {REPO_ROOT}")
            print("Refreshing existing clone ...")
            subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_BRANCH], check=True)
            subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH], check=True)
            subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
    NOTEBOOK_DIR = REPO_ROOT / "notebooks"
    os.chdir(NOTEBOOK_DIR)
    print(f"Changed working directory to {NOTEBOOK_DIR}")
else:
    REPO_ROOT = NOTEBOOK_DIR.parent

PKG_DIR = REPO_ROOT / "vllm_hook_plugins"
REQ_FILE = REPO_ROOT / "requirement.txt"

print("Colab      :", IN_COLAB)
print("Notebook dir:", NOTEBOOK_DIR)
print("Repo root   :", REPO_ROOT)
print("Repo branch :", REPO_BRANCH)
print("Package dir :", PKG_DIR)
print("Req file    :", REQ_FILE)

if IN_COLAB:
    try:
        import torch
    except Exception:
        torch = None
    has_cuda = bool(torch is not None and torch.cuda.is_available())
    has_cudart = importlib.util.find_spec("nvidia.cuda_runtime") is not None
    if not has_cuda and not has_cudart:
        raise RuntimeError(
            "This notebook requires a Colab GPU runtime with CUDA available. "
            "In Colab, use Runtime > Change runtime type > T4 GPU (or another GPU), then rerun from a fresh runtime."
        )

if not PKG_DIR.exists():
    raise FileNotFoundError(f"Plugin directory not found: {PKG_DIR}")

if shutil.which("git") is None and IN_COLAB:
    raise RuntimeError("Colab was detected but git is unavailable in the runtime.")

if REQ_FILE.exists():
    req_cmd = [sys.executable, "-m", "pip", "install", "-r", str(REQ_FILE)]
    print("Running:", " ".join(req_cmd))
    subprocess.run(req_cmd, check=True)
else:
    print("Warning: requirement.txt not found; skipping dependency install.")

protobuf_cmd = [sys.executable, "-m", "pip", "install", "--force-reinstall", "protobuf>=5.29.6,<6.30"]
print("Running:", " ".join(protobuf_cmd))
subprocess.run(protobuf_cmd, check=True)

editable_cmd = [sys.executable, "-m", "pip", "install", "-e", str(PKG_DIR)]
print("Running:", " ".join(editable_cmd))
subprocess.run(editable_cmd, check=True)

plugin_src_dir = str(PKG_DIR.resolve())
if plugin_src_dir not in sys.path:
    sys.path.insert(0, plugin_src_dir)
importlib.invalidate_caches()
print("Plugin source:", plugin_src_dir)
print("Python exec  :", sys.executable)


### Importing the Hook-Enabled LLM
The plugin provides its own LLM wrapper that behaves like vllm.LLM (`from vllm import LLM`) but adds support for hooks and instrumentation.
We import it here:

In [ ]:
from vllm_hook_plugins import HookLLM

### Environment & multiprocessing setup

In [ ]:
import io
import os
import multiprocessing as mp
import sys
import torch
from vllm import SamplingParams

IN_COLAB = "google.colab" in sys.modules
os.environ["VLLM_USE_V1"] = "1"

if IN_COLAB:
    mp.set_start_method("fork", force=True)
    os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "fork"
    os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
    os.environ.setdefault("HF_HOME", "/content/.cache/huggingface")
    os.environ.setdefault("HUGGINGFACE_HUB_CACHE", "/content/.cache/huggingface/hub")
    os.makedirs(os.environ["HUGGINGFACE_HUB_CACHE"], exist_ok=True)
    os.makedirs("/content/.cache/vllm-hook", exist_ok=True)

    def _patch_fileno(stream, fallback_stream, fallback_fd):
        try:
            stream.fileno()
        except io.UnsupportedOperation:
            def _fileno():
                try:
                    return fallback_stream.fileno()
                except Exception:
                    return fallback_fd
            stream.fileno = _fileno

    _patch_fileno(sys.stdout, sys.__stdout__, 1)
    _patch_fileno(sys.stderr, sys.__stderr__, 2)
    print("Colab detected: using fork start method and disabled V1 multiprocessing")
else:
    mp.set_start_method("spawn", force=True)
    os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"


### Initialize `HookLLM`
Before we create the LLM instance, we need to specify the model and data type:

In [ ]:
cache_dir = '/content/.cache/vllm-hook' if 'google.colab' in sys.modules else str(Path('~/.cache/vllm-hook').expanduser())
model = 'ibm-granite/granite-4.0-micro'

dtype_map = {
    'ibm-granite/granite-4.0-micro': torch.float16,
    'microsoft/Phi-3-mini-4k-instruct': 'auto',
}

# Leave quantization off by default; enable only with a supported Granite checkpoint/runtime.
quantization = None
compilation_config = {"mode": 0}


We also need to provide a config file that specifies how activations are steered (e.g., which layers to intervene on, which token to intervene, what direction vectors to apply, etc.).<br />
In the following example, we apply activation steering at the 15th layer, apply the steering at all positions (as opposed to only at the start of the decoding process), and along the direction given in `vector_path`:

In [ ]:
import json
from pathlib import Path

json_path = Path("../model_configs/activation_steer/granite-4.0-micro.json")

with open(json_path, "r") as f:
    config = json.load(f)

# print(config)


Inside `steer_hook_act` we defined the activation steering behavior during model inference.
Now, we initialize the llm:

In [ ]:
llm = HookLLM(
    model=model,
    worker_name="steer_hook_act",
    config_file=json_path,
    download_dir=cache_dir,
    trust_remote_code=True,
    dtype=dtype_map[model],
    enable_hook=True,

    ### Memory Conservation Args (Important for Colab T4 instance)

    # Raise GPU reservation enough to leave room for model weights and a small KV cache.
    gpu_memory_utilization=0.3,

    # Cap context length to fit KV cache on the free public Colab T4 instance.
    max_model_len=1024,

    # Limit batch concurrency because a single free public Colab T4 has tight memory headroom.
    max_num_seqs=1,

    # Keep eager execution explicit so CUDA graph capture does not add memory overhead.
    enforce_eager=True,

    # Keep compilation settings conservative for T4 memory pressure.
    compilation_config=compilation_config,

    # Stay on a single T4 device; tensor parallelism does not help on a one-GPU Colab runtime.
    tensor_parallel_size=1,

    # Try 4-bit inflight quantization on the Colab T4.
    quantization=None, # quantization="bitsandbytes",

    # Disable prefix caching to avoid extra memory overhead on constrained T4 sessions.
    enable_prefix_caching=False,
)


### Test case
In the following, we show a test case and compare generations **with** and **without** activation steering.

**Note**: Users should swap the example configs with their own to show desirable performance. The following is for pipeline illustration only.

In [ ]:
test_cases = [
    "Write a dialogue between two people, one is dressed up in a ball gown and the other is dressed down in sweats. The two are going to a nightly event. Your answer must contain exactly 3 bullet points in the markdown format (use \"* \" to indicate each bullet) such as:\n* This is the first point.\n* This is the second point.",
    "What is the difference between the 13 colonies and the other British colonies in North America? Your answer must contain exactly 6 bullet point in Markdown using the following format:\n* Bullet point one.\n* Bullet point two.\n...\n* Bullet point fix."
]

Before we start, we define the sampling parameters:

**Note**: token 32007 is phi-specific, refer to the original huggingface implementation for details https://github.com/microsoft/llm-steer-instruct/blob/main/utils/generation_utils.py.

In [ ]:
sampling_params = SamplingParams(
    temperature=0.0,                       
    max_tokens=2048,
    stop_token_ids=[llm.tokenizer.eos_token_id, 32007],  
)

Next, for each prompt, we:
1. Apply chat template on the test cases
2. Generate with activation steering enabled (`use_hook=True`, default),
3. Reset the prefix cache to ensure the baseline generation does not reuse steered cache,
4. Generate again with `use_hook=False` to obtain the baseline output.

In [ ]:
outputs = []
outputs_original = []

for case in test_cases:
    print("=" * 100)
    prompt = case
    messages = [{"role": "user", "content": prompt}]
    example = llm.tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

    outputs.extend(llm.generate(example, sampling_params))
    
    llm.llm_engine.reset_prefix_cache()
    
    outputs_original.extend(llm.generate(example, sampling_params, use_hook=False))
    
    llm.llm_engine.reset_prefix_cache()

Finally we can print out the results as follows:

In [ ]:
for steered, original in zip(outputs, outputs_original):
    print("=" * 100)
    steered_text = steered.outputs[0].text
    print("\n[With activation steering]\n")
    print(steered_text)
    
    baseline_text = original.outputs[0].text
    print("\n[Without activation steering]\n")
    print(baseline_text)